# CRC Cluster / SLURM / JupyterLab

## Summary
Infrastructure setup for running segmentation and loop-calling pipelines on the CRC HPC cluster — from getting lab access through running a custom-kernel JupyterLab notebook. Written as a guide for future lab members getting started on the cluster.

## Why to Use
Needed for GPU-accelerated segmentation (CellposeSAM) and cluster-scale chromatin loop calling jobs too large/slow for local compute.

## When to Use
Any job requiring GPU access, SLURM job scheduling, or persistent JupyterLab sessions on shared cluster resources.

## How to Use

### 1. Getting Access
- Access was requested on my behalf by PI/Aaron (not a self-service form)
- _Fill in: how long it took to get credentials_

### 2. Allocation
- No separate allocation/group name to request — access is tied to lab storage path and the lab's group account
- Lab storage path: `/ix/apoholek/mxie/`
- Group/allocation account: `apoholek` (confirmed via `sacctmgr list user mex15`, which shows `Def Acct: apoholek`)
- To check your own account/allocation:
```bash
sacctmgr list user <your_username>
```
- To check usage against the allocation's limits:
```bash
crc-usage
```
- _Fill in: storage quota, compute hour limits (if any) — check via `crc-usage` output_

### 3. Connecting to the Cluster

**Via terminal (SSH):**
```bash
ssh mex15@h2p.crc.pitt.edu
# enter password when prompted
```

**Via OnDemand (browser-based JupyterLab):**
1. Turn on VPN
2. Go to OnDemand, select the new cluster
3. Launch JupyterLab app from there

### 4. Point JupyterLab at the Lab's Storage Directory
By default, JupyterLab's root directory won't point at `/ix/apoholek`. To fix this once (persists across sessions):

```bash
# from terminal
jupyter lab --generate-config
nano ~/.jupyter/jupyter_lab_config.py
```

At the top of the config file, add:
```python
c.ServerApp.root_dir = '/ix/apoholek'
```

Save (`Ctrl+O`, Enter) → Exit (`Ctrl+X`) → restart JupyterLab (relaunch the OnDemand session) for it to take effect.

### 5. CPU vs. GPU — When to Use Which
- **GPU required**: CellposeSAM (segmentation)
- **GPU used for speed, not strictly required**: chromatin loop calling jobs (cLoops2, Peakachu, FitHiChIP) — these will run on CPU, but GPU is used for faster compute
- _Fill in: how to request GPU specifically in OnDemand's job form (which dropdown/toggle) or SLURM script flags (`--gres=gpu:1` etc.)_

### 6. Setting Up a New Conda Environment
Each tool gets its own conda env, stored under the lab's storage path (not the default home-directory location):

```bash
module load anaconda3/2025.7.0-2-python_3.11

# create env in default location
conda create -n env_name python=3.10 -y
conda activate env_name

# OR create env at a specific prefix under lab storage (preferred)
conda create --prefix /ix/apoholek/mxie/envs/env_name python=3.11 -y
conda activate /ix/apoholek/mxie/envs/env_name

# to remove an env
conda env remove --prefix /ix/apoholek/mxie/envs/env_name
```

Example envs currently set up:
- `/ix/apoholek/mxie/envs/basic` (python 3.11)
- `/ix/apoholek/mxie/envs/peakachu` (python 3.10)
- `/ix/apoholek/mxie/envs/stardist` (python 3.11)

### 7. Installing StarDist (example walkthrough)
```bash
conda create --prefix /ix/apoholek/mxie/envs/stardist python=3.11 -y
conda activate /ix/apoholek/mxie/envs/stardist
# if conda activate doesn't work, try:
source activate /ix/apoholek/mxie/envs/stardist

pip install tensorflow stardist
# if this errors out:
unset PYTHONPATH
pip install tensorflow==2.15 stardist
```

### 8. Registering a Conda Env as a Jupyter Kernel
```bash
pip install ipykernel
python -m ipykernel install --user --name stardist --display-name "Python (stardist)"
```

**If the kernel disappears after making it**, re-register it:
```bash
conda activate /ix/apoholek/mxie/envs/stardist
python -m ipykernel install --user --name stardist --display-name "Python (stardist)"
conda deactivate
```

## Quirks / Gotchas

- **JupyterLab root directory doesn't default to lab storage** — must manually edit `jupyter_lab_config.py` (see Section 4), or every session starts in the wrong place.

- **`pip install tensorflow stardist` fails** — a lingering `PYTHONPATH` env variable conflicts with the install. Fix: `unset PYTHONPATH` before installing, and pin `tensorflow==2.15`.

- **Kernel vanishes after creating it** — re-run the `ipykernel install` command from inside the activated env (Section 8). Seems to need re-registering more than once in some cases.

- **PYTHONPATH kernel conflicts (general)** — if a kernel is pulling in packages from the wrong environment (python path issue), check and edit its `kernel.json`:
```bash
cat ~/.local/share/jupyter/kernels/<kernel_name>/kernel.json
nano ~/.local/share/jupyter/kernels/<kernel_name>/kernel.json
```
Make sure the `env` block looks like:
```json
"kernel_protocol_version": "5.5",
"env": {
  "PYTHONPATH": "",
  "PYTHONNOUSERSITE": "0"
}
```
This clears out a bad inherited `PYTHONPATH` so the kernel only sees its own env's packages.

- **SSH host key errors**: _fill in fix_
- **Module loading issues**: _fill in which modules conflicted and how resolved_

## Output Interpretation
N/A — infrastructure reference only.